In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
!curl -O $data
df = pd.read_csv('car_fuel_efficiency_2026.csv')


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  620k  100  620k    0     0  4330k      0 --:--:-- --:--:-- --:--:-- 4337k


In [4]:
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


***Variables of interest - engine_displacement, num_cylinders, horsepower, vehicle_weight, acceleration***


**Target Variable - fuel_efficiency_mpg**

In [5]:
## Starting with Data Cleaning
df.columns = df.columns.str.lower().str.replace(' ', '_')
df.dtypes
df_str = df.select_dtypes(include=['object','str'])
df[df_str.columns] = df_str.apply(lambda x: x.str.lower().str.replace(' ', '_'))
df.head()







,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,europe,gasoline,front-wheel_drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,europe,diesel,front-wheel_drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,asia,gasoline,front-wheel_drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,europe,gasoline,front-wheel_drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,usa,diesel,front-wheel_drive,3,2580,7,304.0,4510,17.5,31.0


In [6]:
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,europe,gasoline,front-wheel_drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,europe,diesel,front-wheel_drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,asia,gasoline,front-wheel_drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,europe,gasoline,front-wheel_drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,usa,diesel,front-wheel_drive,3,2580,7,304.0,4510,17.5,31.0


In [7]:
base = ['fuel_type','engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight', 'acceleration','fuel_efficiency_mpg']
df_subset = df[base].copy()
df_subset.head()

## use fuel_type as a catagorical variable
df.fuel_type.value_counts()
## three types of fuel: diesel, gasoline, hybrid
## i need to create three new columns for each type of the fuel and assign 1 or 0 to each row based on the type of fuel
fuel_type = df.fuel_type.unique()
fuel_type
## for fuel in fuel_type: 
## column name would be something like "type_diesel", "type_gasoline", "type_hybrid" --> **f(type_{fuel})**
## I am going to loop through the fuel types and assign a value of 1 if the row matches the fuel type and 0 if it does not match. 

type_diesel = (df_subset.fuel_type == "diesel").astype(int)
df_subset['type_diesel'] = type_diesel
df_subset.head()

for fuel in fuel_type:
    df_subset[f'type_{fuel}'] = (df_subset.fuel_type == fuel).astype(int)

df_subset.head()

,fuel_type,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg,type_diesel,type_gasoline,type_hybrid
0,gasoline,2180,6,243.0,3870,NaN,31.9,0,1,0
1,diesel,2390,6,272.0,4210,NaN,31.3,1,0,0
2,gasoline,2320,6,267.0,4240,17.3,27.5,0,1,0
3,gasoline,2130,6,258.0,4490,18.7,28.5,0,1,0
4,diesel,2580,7,304.0,4510,17.5,31.0,1,0,0


## Next step is to chunk this dataset into training, validation and testing dataset. 
Things to keep in mind are: 
1. remember how the indexing is working
2. rows need to be randomly selected, and need to be fixed each time a dataframe is run
3. make sure that the order of index does not reset when you call a dataframe

df_subset_train
df_subset_val
df_subset_test

How do I start here - 
1. identify the number of rows in the dataframe
2. count them, and create variables to store num of rows you'd want in you train, val and test dataframes. 60% of the rows (fixed) should be assigned to training, 20% to validation, and 20% to testing
3. 

In [8]:
df_subset.drop(columns='fuel_type', inplace=True)
df_subset.describe()
n = len(df_subset)
n_train = int(n*0.6)
n_val = int(n*0.2)
n_test = n - n_train - n_val
n_train, n_val, n_test

idx = np.arange(n)
np.random.seed(2)
np.random.shuffle(idx)
idx



array([7878, 3224, 1919, ..., 6637, 2575, 7336], shape=(10000,))

In [9]:
idx_train = idx[:n_train]
idx_val = idx[n_train:n_train+n_val]
idx_test = idx[n_train+n_val:]

idx_train, idx_val, idx_test

(array([7878, 3224, 1919, ..., 4729, 9352, 9577], shape=(6000,)),
 array([9302, 6652, 4756, ..., 7654, 8741, 6289], shape=(2000,)),
 array([8709, 9991, 7370, ..., 6637, 2575, 7336], shape=(2000,)))

In [10]:

df_subset.head()
df_subset_train = df_subset.iloc[idx_train].reset_index(drop=True)
df_subset_val = df_subset.iloc[idx_val].reset_index(drop=True)
df_subset_test = df_subset.iloc[idx_test].reset_index(drop=True)

df_subset_train.head()


,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg,type_diesel,type_gasoline,type_hybrid
0,2520,7,297.0,4510,17.8,32.0,1,0,0
1,1970,6,237.0,3990,17.9,31.9,0,1,0
2,2250,6,245.0,4230,18.2,28.7,0,1,0
3,2120,6,227.0,4420,20.5,29.9,0,1,0
4,2260,6,264.0,4090,16.9,31.8,1,0,0


In [11]:
## Setting up the Traget Variable

y_train = df_subset_train.fuel_efficiency_mpg.values
y_val = df_subset_val.fuel_efficiency_mpg.values
y_test = df_subset_test.fuel_efficiency_mpg.values

y_train, y_val, y_test

df_subset_train.drop(columns='fuel_efficiency_mpg', inplace=True)
df_subset_val.drop(columns='fuel_efficiency_mpg', inplace=True)
df_subset_test.drop(columns='fuel_efficiency_mpg', inplace=True)

In [12]:
df_subset_train.head()

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,type_diesel,type_gasoline,type_hybrid
0,2520,7,297.0,4510,17.8,1,0,0
1,1970,6,237.0,3990,17.9,0,1,0
2,2250,6,245.0,4230,18.2,0,1,0
3,2120,6,227.0,4420,20.5,0,1,0
4,2260,6,264.0,4090,16.9,1,0,0


In [13]:
## Resolving the null_values in the dataset. 
## Mean is calculated from the Training data only to backfill. 
#df_subset_train[feature_columns].isnull().sum()

#hp_mean = df_subset_train.horsepower.mean()
#acceleration_mean = df_subset_train.acceleration.mean()
#hp_mean, acceleration_mean
#df_subset_train.horsepower = df_subset_train.horsepower.fillna(hp_mean)
#df_subset_train.acceleration = df_subset_train.acceleration.fillna(acceleration_mean)
#df_subset_val.horsepower = df_subset_val.horsepower.fillna(hp_mean)
#df_subset_val.acceleration = df_subset_val.acceleration.fillna(acceleration_mean)
#df_subset_test.horsepower = df_subset_test.horsepower.fillna(hp_mean)
#df_subset_test.acceleration = df_subset_test.acceleration.fillna(acceleration_mean)


In [14]:
df_subset_train.values
df_subset.columns

Index(['engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight',
       'acceleration', 'fuel_efficiency_mpg', 'type_diesel', 'type_gasoline',
       'type_hybrid'],
      dtype='str')

In [15]:
### Setting up the Feature Matrix
# Create a function that can pass any dataframe and prepare the feature matrix for that
# what needs to happen in this function? 
## fill na values 
## turn the dataframe into a numpy array

hp_mean = df_subset_train.horsepower.mean()
acceleration_mean = df_subset_train.acceleration.mean()

feature_columns = ['engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight',
       'acceleration', 'type_diesel', 'type_gasoline',
       'type_hybrid']

def prepare_feature_matrix(df):
    df = df.copy()
    df.horsepower = df.horsepower.fillna(hp_mean)
    df.acceleration = df.acceleration.fillna(acceleration_mean)
    return df[feature_columns].values

In [16]:
df_subset_val[feature_columns].isnull().sum()

engine_displacement      0
num_cylinders            0
horsepower             180
vehicle_weight           0
acceleration            61
type_diesel              0
type_gasoline            0
type_hybrid              0
dtype: int64

In [17]:
X_train = prepare_feature_matrix(df_subset_train)
X_val = prepare_feature_matrix(df_subset_val)
X_test = prepare_feature_matrix(df_subset_test)

X_test.dtype

dtype('float64')

In [18]:
## Defining the linear regression function with regularization parameter.

def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    reg = r * np.eye(XTX.shape[0])
    XTX = XTX + reg

    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    
    return w[0], w[1:]

In [19]:
train_linear_regression_reg(X_train, y_train, r=0.1)

(np.float64(41.50306302220011),
 array([-3.44408764e-05,  1.77843663e-01, -2.56824428e-02, -3.59035961e-03,
        -1.70616208e-01,  1.38821045e+01,  1.18014651e+01,  1.58194934e+01]))

In [20]:
train_linear_regression_reg(X_train, y_train, r=0.0)

(np.float64(108.248193359375),
 array([ 4.92558102e-02, -3.01845179e+01, -3.16060740e-02, -3.28703990e-03,
        -1.45532803e-01,  1.56534424e+01,  1.33091919e+01,  2.27220389e+01]))

In [21]:
## Marking predictions

w0, w1 = train_linear_regression_reg(X_train, y_train, r=0.01)
y_pred = w0 + X_val.dot(w1)
y_pred

array([30.96716563, 29.25405925, 28.80005133, ..., 32.97069491,
       28.79256717, 27.60480299], shape=(2000,))

In [22]:
## Calculating RMSE
def rmse(y, y_pred):
    error = y_pred - y
    mse = (error ** 2).mean()
    return np.sqrt(mse)

rmse(y_val, y_pred)

np.float64(2.2124355045841435)

In [ ]:
## Picking the best regularization parameter
r_values = [0.0, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

for r in r_values: 
    w0, w1 = train_linear_regression_reg(X_train, y_train, r)
    y_pred = w0 + X_val.dot(w1)
    score = rmse(y_val, y_pred)
    print(f"RMSE for r={r}: {score:.3f}")




RMSE for r=0.0: 12.669
RMSE for r=0.01: 2.212
RMSE for r=0.1: 2.212
RMSE for r=1.0: 2.211
RMSE for r=10.0: 2.366
RMSE for r=100.0: 3.081
RMSE for r=1000.0: 3.379


In [26]:
X_train_val = np.concatenate([X_train, X_val])
y_train_val = np.concatenate([y_train, y_val])

X_train_val.shape, y_train_val.shape

((8000, 8), (8000,))

In [29]:
w0, w1 = train_linear_regression_reg(X_train_val, y_train_val, r=1.0)
y_pred = w0 + X_test.dot(w1)
score = rmse(y_test, y_pred)
score
    




np.float64(2.3218189035130568)